# 05 — Day 10: Uniform vs Credibility-Weighted Macro-F1 on Fin-Fact

Tiny-sample (n=20) eval to validate the eval harness. Full 1304-claim run is queued for a later session — wall time on M4 with glm-4.7-flash is ~25s/claim → ~9 hours for the full pair of runs.

**Plan-stated success criterion (Day 10):** `weighted Macro-F1 ≥ uniform Macro-F1`. If they tie, that is consistent with the eval being valid; if weighted < uniform, the rank-decay constants in `arxivmind_adapter.py` are off and need re-tuning.

**Honest result on n=20:** both modes produce 100% ABSTAIN. Cause: Fin-Fact gold claims are US-political (Biden, Convoy, Massachusetts) while our Day-2 news index is 2026-05-14 UK/EU financial. The reasoner *correctly* abstains because retrieved evidence is irrelevant — a different failure than misclassification. Next iteration must align the news index with the eval corpus.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))

from crag_fin.eval.metrics import compute_finfact_metrics  # noqa: E402

RUNS_ROOT = ROOT / 'experiments' / 'runs' / 'run_001_uniform_vs_weighted'

rows = []
for mode in ['uniform', 'weighted']:
    m = compute_finfact_metrics(RUNS_ROOT / f'{mode}_n20' / 'predictions.jsonl')
    rows.append((mode, m))
    print(f'\n=== {mode.upper()} (n={m["n"]}) ===')
    print(f'  macro_f1:        {m["macro_f1"]:.3f}')
    print(f'  accuracy:        {m["accuracy"]:.3f}')
    print(f'  abstention_rate: {m["abstention_rate"]:.3f}')
    print(f'  gold dist:       {m["gold_dist"]}')
    print(f'  pred dist:       {m["pred_dist"]}')

## Diagnostic: why 100% abstention?

We dump the first 3 (claim, top-2-retrieved-sources) pairs to show the corpus mismatch directly.

In [2]:
import json

with open(RUNS_ROOT / 'uniform_n20' / 'predictions.jsonl') as f:
    for i, line in enumerate(f, 1):
        if i > 3:
            break
        rec = json.loads(line)
        print(f'\n[{i}] gold={rec["gold"]:<6} pred={rec["pred"]}')
        print(f'    claim: {rec["claim"][:90]}')
        if rec['propositions']:
            srcs = rec['propositions'][0]['passage_sources'][:2]
            print('    top retrieved sources for first proposition:')
            for s in srcs:
                print(f'      {s[:80]}')


[1] gold=false  pred=ABSTAIN
    claim: Did Biden Call for an 'End to Shareholder Capitalism'?
    top retrieved sources for first proposition:
      https://www.wsj.com/articles/building-products-distributor-qxo-launching-hostile
      https://www.theguardian.com/commentisfree/2026/may/13/the-guardian-view-on-the-k

[2] gold=NEI    pred=ABSTAIN
    claim: Does Jared Kushner's brother own a business associated with COVID-19 testing?
    top retrieved sources for first proposition:
      https://www.cnbc.com/2026/05/14/hantavirus-cruise-ship-outbreak-where-is-it-what
      https://www.theguardian.com/business/2026/may/13/lab-testing-group-intertek-to-b

[3] gold=false  pred=ABSTAIN
    claim: Texas this fiscal year will have more money in reserve than the other 49 states combined.
    top retrieved sources for first proposition:
      https://www.theguardian.com/business/live/2026/may/14/uk-gdp-report-economic-gro
      https://www.telegraph.co.uk/business/2026/05/13/uk-borrowing-gilts

## Day 10 verdict

- Eval harness runs end-to-end, produces structured per-claim JSON, computes Macro-F1 / accuracy / abstention rate correctly (4 unit tests passing in `tests/test_metrics.py`).
- On n=20, both modes tie at Macro-F1 = 0.133 because the retrieved evidence is uniformly off-topic — reasoner abstains correctly. The result is *valid*, just *degenerate*.
- The publication-quality Day-10 deliverable requires a corpus aligned with Fin-Fact claims. Two paths: (a) seed ChromaDB from Fin-Fact's own evidence URLs in the JSON (each entry has an `evidence` field); (b) live-fetch via Tavily per claim (Day 6 tool is wired).
- Hand-off: queue the full 1304-claim run after the corpus alignment fix. Current harness is reusable; just bump `--n 1304` and pre-seed the right corpus.